In [ ]:
# Step 1: Download processed dataset from shared Drive
import gdown, zipfile, os

if not os.path.exists('dataset/processed'):
    FILE_ID = '1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6'
    gdown.download(id=FILE_ID, output='processed.zip', quiet=False)
    with zipfile.ZipFile('processed.zip', 'r') as z:
        z.extractall('dataset/')
    os.remove('processed.zip')
else:
    print('Already downloaded.')

In [ ]:
# Step 2: Load shared splits and ID mappings — use these in every notebook, never recreate them
import pandas as pd
import numpy as np
import pickle

train_df = pd.read_csv('dataset/processed/train.csv')
val_df   = pd.read_csv('dataset/processed/val.csv')
test_df  = pd.read_csv('dataset/processed/test.csv')

with open('dataset/processed/id_maps.pkl', 'rb') as f:
    maps = pickle.load(f)

n_users = len(maps['user2idx'])
n_items = len(maps['item2idx'])

print(f'n_users: {n_users:,} | n_items: {n_items:,}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
# Step 3: Build training data from train_df — never use user_ratings_cleaned.csv directly
user_positives = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

#from here: build your model, do any model-specific preprocessing, train, then call evaluate() at the end

In [ ]:
# Step 4: Shared evaluation function — copy this into every notebook as-is
def evaluate(score_series, train_df, test_df, n_negatives=99, K=10, seed=42):
    rng = np.random.default_rng(seed)
    all_items = np.array(list(maps['item2idx'].values()))
    seen = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

    hits, ndcgs, mrrs = [], [], []

    for _, row in test_df.iterrows():
        uid      = int(row['user_idx'])
        pos_item = int(row['item_idx'])

        user_seen  = seen.get(uid, set()) | {pos_item}
        candidates = np.setdiff1d(all_items, list(user_seen))
        neg_items  = rng.choice(candidates, size=n_negatives, replace=False)

        items_to_rank = np.append(neg_items, pos_item)
        scores = score_series.reindex(items_to_rank).fillna(0).values
        ranked = items_to_rank[np.argsort(-scores)]

        pos_rank = np.where(ranked == pos_item)[0][0] + 1

        hits.append(1 if pos_rank <= K else 0)
        ndcgs.append(np.log(2) / np.log(pos_rank + 1) if pos_rank <= K else 0)
        mrrs.append(1 / pos_rank if pos_rank <= K else 0)

    return {
        f'HR@{K}':   round(np.mean(hits), 4),
        f'NDCG@{K}': round(np.mean(ndcgs), 4),
        f'MRR@{K}':  round(np.mean(mrrs), 4),
    }

In [ ]:
# Step 5: How to call evaluate() with your model
# Your model should produce a score for every item for a given user
# Wrap those scores as a pd.Series indexed by item_idx and pass to evaluate()

# Example for any model that gives a scores array of shape (n_items,) per user:
#
# all_scores = {}  # dict of {user_idx: scores_array}
# for uid in test_df['user_idx'].unique():
#     scores_array = model.get_scores(uid)  # your model's scoring function
#     all_scores[uid] = scores_array
#
# Then for a global score series (like popularity):
# score_series = pd.Series(scores_array, index=range(n_items))
# results = evaluate(score_series, train_df, test_df)
# print(results)